# MVTec Anomaly Detection — All Categories
**Pipeline:** Preprocessing → Harris → Pyramid → SIFT → Segmentation → Classification

Select a category from the Gradio interface and everything runs automatically.

**Dataset:** `/kaggle/input/mvtec-ad/`

## 0. Setup & Imports

In [91]:
import os
import glob
import numpy as np

# ── Matplotlib backend لازم يتعمل قبل أي import تاني ─────────────────────
import matplotlib
matplotlib.use('Agg')          # بدون GUI — مهم جداً في Kaggle + Gradio
import matplotlib.pyplot as plt

import cv2
import io
from PIL import Image
import gradio as gr

from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import copy
import warnings
warnings.filterwarnings('ignore')

# ── Paths & Config ────────────────────────────────────────────────────────
# عدّلي المسار لو الداتاست في مكان تاني
DATASET_ROOT = '/kaggle/input/datasets/ipythonx/mvtec-ad'

ALL_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill',
    'screw', 'tile', 'toothbrush', 'transistor',
    'wood', 'zipper'
]

IMG_SIZE = (256, 256)
SEED     = 42
np.random.seed(SEED)

sift = cv2.SIFT_create(nfeatures=300)

# ── Global state for Gradio ───────────────────────────────────────────────
_state = {
    'category'  : None,
    'clf'       : None,
    'scaler'    : None,
    'results'   : None,
    'best_name' : None,
    'defects'   : [],
}

print('✓ Imports & Setup done')

✓ Imports & Setup done


## 1. Core Pipeline Functions

In [95]:
# ─────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────
def load_images(folder, label):
    paths = sorted(glob.glob(os.path.join(folder, '*.png')))
    imgs, labels = [], []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        imgs.append(img)
        labels.append(label)
    return imgs, labels


def load_category(category):
    base       = os.path.join(DATASET_ROOT, category)
    train_good = os.path.join(base, 'train', 'good')
    test_dir   = os.path.join(base, 'test')

    if not os.path.isdir(train_good):
        raise FileNotFoundError(f'مسار التدريب مش موجود: {train_good}')

    train_imgs, train_labels = load_images(train_good, label=0)

    test_imgs, test_labels, defect_types = [], [], []
    if os.path.isdir(test_dir):
        defect_types = sorted([
            d for d in os.listdir(test_dir)
            if os.path.isdir(os.path.join(test_dir, d))
        ])
        for dt in defect_types:
            lbl  = 0 if dt == 'good' else 1
            imgs, lbls = load_images(os.path.join(test_dir, dt), label=lbl)
            test_imgs  += imgs
            test_labels += lbls

    return train_imgs, train_labels, test_imgs, test_labels, defect_types


# ─────────────────────────────────────────────────────────────────────────
# PREPROCESSING & FEATURES
# ─────────────────────────────────────────────────────────────────────────
def harris_corners(img_rgb, threshold=0.01):
    gray     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    response = cv2.cornerHarris(gray, blockSize=2, ksize=3, k=0.04)
    response = cv2.dilate(response, None)
    marked   = img_rgb.copy()
    marked[response > threshold * response.max()] = [255, 0, 0]
    count    = int(np.sum(response > threshold * response.max()))
    return marked, count


def kmeans_segment(img_rgb, k=3):
    pixels   = img_rgb.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS
    )
    centers  = centers.astype(np.uint8)
    seg_img  = centers[labels.flatten()].reshape(img_rgb.shape)
    lbl_mask = labels.reshape(img_rgb.shape[:2])
    return seg_img, lbl_mask


def extract_features(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gray_u8 = gray.astype(np.uint8)

    gauss  = cv2.GaussianBlur(gray, (5, 5), 1.5)
    median = cv2.medianBlur(gray_u8, 5).astype(np.float32)
    diff_g = np.abs(gray - gauss)
    diff_m = np.abs(gray - median)

    response       = cv2.cornerHarris(gray, 2, 3, 0.04)
    corner_density = float(np.sum(response > 0.01 * response.max())) / (IMG_SIZE[0] * IMG_SIZE[1])

    kps, descs = sift.detectAndCompute(gray_u8, None)
    n_kps      = len(kps)
    resp_mean  = float(np.mean([k.response for k in kps])) if kps else 0.0
    resp_std   = float(np.std ([k.response for k in kps])) if kps else 0.0
    desc_mean  = float(descs.mean()) if descs is not None else 0.0
    desc_std   = float(descs.std())  if descs is not None else 0.0

    lap      = cv2.Laplacian(gray_u8, cv2.CV_64F)
    lap_mean = float(np.mean(np.abs(lap)))
    lap_std  = float(np.std(lap))
    lap_var  = float(lap.var())

    r_mean, g_mean, b_mean = img_rgb[:, :, 0].mean(), img_rgb[:, :, 1].mean(), img_rgb[:, :, 2].mean()
    r_std,  g_std,  b_std  = img_rgb[:, :, 0].std(),  img_rgb[:, :, 1].std(),  img_rgb[:, :, 2].std()

    _, lbl     = kmeans_segment(img_rgb, k=4)
    counts     = np.bincount(lbl.flatten(), minlength=4)[:3]
    seg_ratios = sorted([c / lbl.size for c in counts])

    return np.array([
        diff_g.mean(), diff_g.std(), diff_m.mean(), diff_m.std(),    
        corner_density,                                              
        n_kps, resp_mean, resp_std, desc_mean, desc_std,             
        lap_mean, lap_std, lap_var,                                  
        r_mean, g_mean, b_mean, r_std, g_std, b_std,                 
        *seg_ratios                                                  
    ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────
# TRAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────
def train_pipeline(category):
    train_imgs, train_labels, test_imgs, test_labels, defect_types = load_category(category)

    if len(train_imgs) == 0:
        raise ValueError(f'مفيش صور في train/good لـ {category}')

    all_imgs   = train_imgs + test_imgs
    all_labels = train_labels + test_labels

    X = np.array([extract_features(img) for img in all_imgs])
    y = np.array(all_labels)

    stratify_y = y if len(np.unique(y)) > 1 else None

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=SEED, stratify=stratify_y
    )

    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_te_s   = scaler.transform(X_te)

    classifiers = {
        'Naive Bayes'      : GaussianNB(),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
        'Random Forest'    : RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=SEED),
    }

    results = {}
    trained_clfs = {}
    for name, clf in classifiers.items():
        clf.fit(X_tr_s, y_tr)
        y_pred = clf.predict(X_te_s)
        results[name] = {
            'acc' : accuracy_score (y_te, y_pred),
            'prec': precision_score(y_te, y_pred, zero_division=0),
            'rec' : recall_score   (y_te, y_pred, zero_division=0),
            'f1'  : f1_score       (y_te, y_pred, zero_division=0),
            'cm'  : confusion_matrix(y_te, y_pred),
        }
        trained_clfs[name] = clf

    best_name = max(results, key=lambda k: results[k]['f1'])
    
    best_clf = copy.deepcopy(trained_clfs[best_name].__class__(
        **trained_clfs[best_name].get_params()
    ))
    X_all_s = scaler.fit_transform(X)
    best_clf.fit(X_all_s, y)

    return best_clf, scaler, results, best_name, defect_types


# ─────────────────────────────────────────────────────────────────────────
# GRADIO HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────
def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig)
    return img

def train_on_category(category):
    if not category: return 'Please select a category first!', None, None
    try:
        clf, scaler, results, best_name, defects = train_pipeline(category)
    except Exception as e: return f'Error during training:\n{e}', None, None

    _state.update({'category': category, 'clf': clf, 'scaler': scaler, 'results': results, 'best_name': best_name, 'defects': defects})

    fig1, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig1.suptitle(f'Classifier Performance — {category}', fontsize=13, fontweight='bold')
    colors  = ['#4C72B0', '#DD8452', '#55A868']
    metrics = ['acc', 'prec', 'rec', 'f1']
    for ax, (cname, color) in zip(axes, zip(results.keys(), colors)):
        r = results[cname]; vals = [r[m] for m in metrics]
        bars = ax.bar(['Accuracy', 'Precision', 'Recall', 'F1'], vals, color=color, alpha=0.85, edgecolor='white')
        ax.set_ylim(0, 1.12); ax.set_title(cname, fontsize=9, fontweight='bold')
        for bar, val in zip(bars, vals): ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout(); metrics_img = fig_to_pil(fig1)

    fig2, axes2 = plt.subplots(1, 3, figsize=(12, 4))
    fig2.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
    for ax, (cname, color) in zip(axes2, zip(results.keys(), colors)):
        cm = results[cname]['cm']; ax.imshow(cm, cmap='Blues'); ax.set_title(cname, fontsize=9)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(['Normal', 'Defect']); ax.set_yticklabels(['Normal', 'Defect'])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
    plt.tight_layout(); cm_img = fig_to_pil(fig2)

    r = results[best_name]
    status = (
        f'✓ Model trained on: {category}\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'Defect types  : {defects}\n\nBest model    : {best_name}\n'
        f'  Accuracy    : {r["acc"]:.4f}\n  Precision   : {r["prec"]:.4f}\n'
        f'  Recall      : {r["rec"]:.4f}\n  F1 Score    : {r["f1"]:.4f}\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\nNow upload an image and click Analyze!'
    )
    return status, metrics_img, cm_img


def analyze_image(pil_image):
    if pil_image is None: return None, 'Please upload an image first!'
    if _state['clf'] is None: return None, 'Please train the model first!'

    try:
        img_rgb = np.array(pil_image.convert('RGB')); img_rgb = cv2.resize(img_rgb, IMG_SIZE)
        feat = extract_features(img_rgb).reshape(1, -1); feat_s = _state['scaler'].transform(feat)
        pred = _state['clf'].predict(feat_s)[0]
        proba = _state['clf'].predict_proba(feat_s)[0] if hasattr(_state['clf'], 'predict_proba') else None

        label_str = 'NORMAL ✓' if pred == 0 else 'DEFECTIVE ✗'; color_hex = '#2d8a4e' if pred == 0 else '#c0392b'
        confidence = f'{max(proba)*100:.1f}%' if proba is not None else 'N/A'

        seg, _ = kmeans_segment(img_rgb, k=3); harris_m, cnt = harris_corners(img_rgb)
        gray_u8 = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY); kps_img, _ = sift.detectAndCompute(gray_u8, None)
        sift_drawn = cv2.drawKeypoints(gray_u8, kps_img, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

        fig, axes = plt.subplots(1, 4, figsize=(18, 4)); fig.patch.set_facecolor('#f8f9fa')
        fig.suptitle(f'Category: {_state["category"]}  |  {label_str}  |  Confidence: {confidence}', fontsize=13, fontweight='bold', color=color_hex, y=1.03)
        axes[0].imshow(img_rgb); axes[0].set_title('Input Image'); axes[0].axis('off')
        axes[1].imshow(seg); axes[1].set_title('K-Means Segmentation (K=3)'); axes[1].axis('off')
        axes[2].imshow(harris_m); axes[2].set_title(f'Harris Corners ({cnt})'); axes[2].axis('off')
        axes[3].imshow(sift_drawn, cmap='gray'); axes[3].set_title(f'SIFT Keypoints ({len(kps_img)})'); axes[3].axis('off')
        plt.tight_layout(); result_img = fig_to_pil(fig)

        proba_str = f'  P(Normal)    = {proba[0]*100:.1f}%\n  P(Defective) = {proba[1]*100:.1f}%\n' if proba is not None else ''
        report = (
            f'============================================\n  ANOMALY DETECTION REPORT\n============================================\n'
            f'  Category     : {_state["category"]}\n  Prediction   : {label_str}\n  Confidence   : {confidence}\n{proba_str}'
            f'  Harris corners : {cnt}\n  SIFT keypoints : {len(kps_img)}\n  Best model     : {_state["best_name"]}\n============================================'
        )
        return result_img, report
    except Exception as e: return None, f'Error during analysis:\n{e}'


def detect_defects(pil_image):
    if pil_image is None: return None, 'Please upload an image first!'
    if _state['clf'] is None: return None, 'Please train the model first!'

    try:
        img_rgb = np.array(pil_image.convert('RGB'))
        img_rgb = cv2.resize(img_rgb, IMG_SIZE)
        gray_u8 = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        gray_f = gray_u8.astype(np.float32)

        # 1. UNIFIED CLASSIFICATION
        # Tab 1 and Tab 2 now share the exact same prediction logic. No overrides.
        feat = extract_features(img_rgb).reshape(1, -1)
        feat_s = _state['scaler'].transform(feat)
        pred = _state['clf'].predict(feat_s)[0]
        proba = _state['clf'].predict_proba(feat_s)[0] if hasattr(_state['clf'], 'predict_proba') else None

        label_str = 'DEFECTIVE ✗' if pred == 1 else 'NORMAL ✓'
        color_hex = '#c0392b' if pred == 1 else '#2d8a4e'
        confidence = f'{max(proba)*100:.1f}%' if proba is not None else 'N/A'

        # 2. Extract Object Mask (Strict Edge Exclusion)
        blurred_for_mask = cv2.GaussianBlur(gray_u8, (11, 11), 0)
        _, fg_mask = cv2.threshold(blurred_for_mask, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Invert mask if background is bright
        corners = [gray_u8[0,0], gray_u8[0,-1], gray_u8[-1,0], gray_u8[-1,-1]]
        if np.mean(corners) > 127: 
            fg_mask = cv2.bitwise_not(fg_mask)
            
        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            fg_mask = np.zeros_like(fg_mask)
            cv2.drawContours(fg_mask, [largest_contour], -1, 255, -1)
            
        # Erode mask deeply to ignore the confusing outer edge boundaries of objects
        inner_mask = cv2.erode(fg_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21)))

        # 3. Generating a Composite Score Map
        # Combination of Blur Difference and Laplacian for texture anomalies
        bg_blur = cv2.GaussianBlur(gray_f, (21, 21), 0)
        diff_map = np.abs(gray_f - bg_blur)
        lap_map = np.abs(cv2.Laplacian(gray_u8, cv2.CV_32F, ksize=3))
        
        score_map = (diff_map * 0.7) + (lap_map * 0.3)
        score_map[inner_mask == 0] = 0 # Ensure we only care about the safe inside of the object

        # Normalize Map for Visualization
        max_val = score_map.max()
        if max_val > 0:
            score_u8 = np.clip(score_map * (255.0 / max_val), 0, 255).astype(np.uint8)
        else:
            score_u8 = np.zeros_like(gray_u8)

        heatmap = cv2.cvtColor(cv2.applyColorMap(score_u8, cv2.COLORMAP_INFERNO), cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(img_rgb, 0.50, heatmap, 0.50, 0)

        annotated = img_rgb.copy()
        defect_regions = []

        def draw_box(img, x, y, w, h, color=(220, 30, 30), thickness=2):
            cv2.rectangle(img, (x, y), (x + w, y + h), color, thickness)
            cl = max(8, int(min(w, h) * 0.20)); pts = [(x, y), (x+w, y), (x, y+h), (x+w, y+h)]; dirs = [(1,1), (-1,1), (1,-1), (-1,-1)]
            for (px, py), (dx, dy) in zip(pts, dirs):
                cv2.line(img, (px, py), (px + dx*cl, py), color, thickness + 2)
                cv2.line(img, (px, py), (px, py + dy*cl), color, thickness + 2)

        # 4. GUARANTEED DYNAMIC BOUNDING BOXES
        if pred == 1 and max_val > 0:
            # If the ML says it's defective, we dynamically find the brightest anomaly in THIS image
            # This guarantees that we ALWAYS box the defect, regardless of absolute intensity
            dynamic_thresh = max_val * 0.40 # Target the top 60% of the localized anomaly
            
            _, binary = cv2.threshold(score_map, dynamic_thresh, 255, cv2.THRESH_BINARY)
            binary = binary.astype(np.uint8)
            
            # Morphological operations to group broken pixels into solid defect areas
            binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))
            binary = cv2.dilate(binary, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
            
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            min_area = IMG_SIZE[0] * IMG_SIZE[1] * 0.0005 
            max_area = IMG_SIZE[0] * IMG_SIZE[1] * 0.40 # Protect against whole-image boxes

            scored = []
            for cnt_c in contours:
                if min_area <= cv2.contourArea(cnt_c) <= max_area:
                    mask_c = np.zeros(score_map.shape, np.uint8)
                    cv2.drawContours(mask_c, [cnt_c], -1, 255, -1)
                    # Score region relative to the overall max anomaly
                    rel_score = float(score_map[mask_c > 0].max()) / max_val
                    scored.append((rel_score, cnt_c))
            
            # Fallback: if no contours survived the filter, force a box on the absolute peak
            if not scored:
                _, _, _, max_loc = cv2.minMaxLoc(score_map)
                cx, cy = max_loc
                w, h = 30, 30
                scored.append((1.0, np.array([[[cx-w, cy-h]], [[cx+w, cy-h]], [[cx+w, cy+h]], [[cx-w, cy+h]]])))

            scored.sort(key=lambda x: x[0], reverse=True)
            kept = scored[:8] 

            for region_score, cnt_c in kept:
                x, y, w, h = cv2.boundingRect(cnt_c)
                
                # Double check the box isn't secretly a giant square
                if (w * h) <= max_area:
                    defect_regions.append((x, y, w, h, region_score))
                    
                    intensity = min(1.0, region_score)
                    box_color = (220, int(30 + (1 - intensity) * 160), 30)
                    
                    draw_box(annotated, x, y, w, h, color=box_color, thickness=2)
                    
                    label_txt = f'DEFECT'
                    (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
                    lx, ly = x, max(y - 4, th + 2)
                    cv2.rectangle(annotated, (lx, ly - th - 2), (lx + tw + 4, ly + 2), box_color, -1)
                    cv2.putText(annotated, label_txt, (lx + 2, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

            banner = f'{label_str}  ({len(defect_regions)} region(s))'
            cv2.rectangle(annotated, (0, 0), (len(banner)*9 + 8, 28), (220, 30, 30), -1)
            cv2.putText(annotated, banner, (4, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
        else:
            cv2.rectangle(annotated, (0, 0), (len(label_str)*9 + 8, 28), (45, 138, 78), -1)
            cv2.putText(annotated, label_str, (4, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

        # Plotting
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.patch.set_facecolor('#f8f9fa')
        fig.suptitle(f'Defect Detection  |  Category: {_state["category"]}  |  {label_str}', fontsize=12, fontweight='bold', color=color_hex)
        axes[0].imshow(img_rgb); axes[0].set_title('Input Image'); axes[0].axis('off')
        axes[1].imshow(overlay); axes[1].set_title('Dynamic Anomaly Heatmap'); axes[1].axis('off')
        axes[2].imshow(annotated); axes[2].set_title('Localized Defect Areas'); axes[2].axis('off')

        plt.colorbar(plt.cm.ScalarMappable(cmap='inferno', norm=plt.Normalize(0, 1)), ax=axes[1], fraction=0.046, pad=0.04).set_label('Relative Heat')
        plt.tight_layout()
        result_pil = fig_to_pil(fig)

        proba_str = f'  P(Normal)      = {proba[0]*100:.1f}%\n  P(Defective)   = {proba[1]*100:.1f}%\n' if proba is not None else ''
        region_str = f'  Defect regions : {len(defect_regions)} area(s) detected\n' + ''.join([f'    Region {idx}: x={x}, y={y}, w={w}, h={h}\n' for idx, (x, y, w, h, rs) in enumerate(defect_regions, 1)]) if defect_regions else '  Defect regions : None detected\n'

        report = (
            f'============================================\n  DEFECT DETECTION REPORT\n============================================\n'
            f'  Category       : {_state["category"]}\n  Verdict        : {label_str}\n  Confidence     : {confidence}\n{proba_str}'
            f'  Peak Intensity : {max_val:.1f}\n{region_str}'
            f'  Model used     : {_state["best_name"]} (Unified)\n  Signals used   : Relative Dynamic Thresholding\n                   + Gaussian Residuals\n============================================'
        )
        return result_pil, report

    except Exception as e: 
        import traceback
        return None, f'Error during defect detection:\n{traceback.format_exc()}'

print('✓ Core logic and helper functions loaded')

✓ Core logic and helper functions loaded


## 2. Gradio Interface

In [96]:
!pip install gradio -q

In [97]:
# ─────────────────────────────────────────────────────────────────────────
# GRADIO INTERFACE LAYOUT
# ─────────────────────────────────────────────────────────────────────────
with gr.Blocks(title='MVTec AD — Multi-Category Anomaly Detector', theme=gr.themes.Soft()) as demo:
    gr.Markdown("# MVTec Anomaly Detection — All Categories\n**Steps:**\n1. Select a category\n2. Click **Train Model**\n3. Go to **Analyze Image** or **Defect Detection**")

    with gr.Row():
        with gr.Column(scale=2):
            cat_dropdown = gr.Dropdown(choices=ALL_CATEGORIES, label='Select Category', value='bottle')
            train_btn = gr.Button('Train Model', variant='primary', size='lg')
        with gr.Column(scale=3):
            train_status = gr.Textbox(label='Training Status', lines=10, placeholder='Select a category and click Train...')

    with gr.Row():
        metrics_out = gr.Image(label='Classifier Metrics')
        cm_out      = gr.Image(label='Confusion Matrices')

    gr.Markdown('---')

    with gr.Tabs():
        with gr.TabItem('Analyze Image'):
            gr.Markdown('### Upload an image to run the full analysis pipeline')
            with gr.Row():
                with gr.Column(scale=1):
                    inp_img     = gr.Image(type='pil', label='Upload image from test/', height=280)
                    analyze_btn = gr.Button('Analyze Image', variant='secondary', size='lg')
                with gr.Column(scale=2):
                    out_vis  = gr.Image(label='Visual Analysis', height=350)
                    out_text = gr.Textbox(label='Detection Report', lines=12, show_copy_button=True)

        with gr.TabItem('Defect Detection'):
            gr.Markdown("### Defect Detection — Localize anomalous regions\nUpload an image and the model will:\n- Classify it as **Normal** or **Defective**\n- Generate a pixel-level **anomaly heatmap**\n- Draw **bounding boxes** around detected defect regions")
            with gr.Row():
                with gr.Column(scale=1):
                    dd_img = gr.Image(type='pil', label='Upload image', height=280)
                    dd_btn = gr.Button('Detect Defects', variant='primary', size='lg')
                    gr.Markdown("**Heatmap legend:**\n- 🔵 Blue → low anomaly score\n- 🔴 Red  → high anomaly score")
                with gr.Column(scale=2):
                    dd_vis    = gr.Image(label='Defect Detection Result', height=380)
                    dd_report = gr.Textbox(label='Defect Detection Report', lines=14, show_copy_button=True)

    # Event wiring
    train_btn.click(fn=train_on_category, inputs=[cat_dropdown], outputs=[train_status, metrics_out, cm_out])
    analyze_btn.click(fn=analyze_image, inputs=[inp_img], outputs=[out_vis, out_text])
    dd_btn.click(fn=detect_defects, inputs=[dd_img], outputs=[dd_vis, dd_report])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7885
* Running on public URL: https://eb454cc3e8b2f47189.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error